# FraudShield: Credit Card Fraud Detection (ML Lab Mini Project)
Step-by-step notebook: **EDA -> split -> 3 models -> imbalance handling -> evaluation -> save model**.
This notebook is self-contained, so it also runs on Google Colab after you upload `creditcard.csv`.
The full pipeline (all strategies, figures, dashboard artefacts) is in `src/train.py`.

## 0. Setup

In [ ]:
# !pip install -q imbalanced-learn   # uncomment on Colab
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, average_precision_score,
                             roc_auc_score, precision_recall_curve, roc_curve)
from imblearn.pipeline import Pipeline            # imblearn Pipeline applies SMOTE to training folds only
from imblearn.over_sampling import SMOTE
sns.set_theme(style="whitegrid")

CANDIDATES = [Path("../data/creditcard.csv"), Path("data/creditcard.csv"), Path("creditcard.csv"),
              Path("../data/creditcard_demo.csv"), Path("data/creditcard_demo.csv")]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH, "Put creditcard.csv in data/ (or upload it to Colab)"
print("Using:", DATA_PATH)
if "demo" in DATA_PATH.name:
    print("WARNING: synthetic demo data - results are NOT valid for your report")

## 1. Load and explore the data (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
df.head()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
counts = df["Class"].value_counts()
print(counts)
print(f"Fraud share: {df['Class'].mean():.4%}")
print(f"Accuracy of a 'predict everything legitimate' model: {1 - df['Class'].mean():.4%}  <- why accuracy is misleading")
sns.countplot(data=df, x="Class"); plt.yscale("log")
plt.title("Legitimate (0) vs Fraudulent (1) transactions - log scale"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
d = df.assign(logAmount=np.log1p(df["Amount"]))
sns.boxplot(data=d, x="Class", y="logAmount", ax=ax[0]); ax[0].set_title("log(1+Amount) by class")
corr = df.drop(columns="Class").corrwith(df["Class"]).sort_values(key=abs, ascending=False).head(12)
corr[::-1].plot.barh(ax=ax[1], title="Top features correlated with Class"); plt.tight_layout(); plt.show()

## 2. Stratified train / validation / test split
The test set stays untouched until the final evaluation.

In [ ]:
X, y = df.drop(columns="Class"), df["Class"]
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval)
print(f"train {len(X_train):,} (fraud {y_train.sum()}) | val {len(X_val):,} (fraud {y_val.sum()}) | test {len(X_test):,} (fraud {y_test.sum()})")

## 3. Preprocessing + three models
Scaling lives inside the pipeline, so it is fitted on training data only. Class weights handle imbalance.

In [ ]:
def prep():
    return ColumnTransformer([("scale", RobustScaler(), ["Time", "Amount"])], remainder="passthrough")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=2,
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=42),
}
pipes = {n: Pipeline([("prep", prep()), ("clf", m)]).fit(X_train, y_train) for n, m in models.items()}
print("trained:", list(pipes))

## 4. Evaluation function (precision, recall, F1, confusion matrix, PR-AUC, ROC-AUC)

In [ ]:
def evaluate_model(model, X_, y_, threshold=0.5, show=True):
    proba = model.predict_proba(X_)[:, 1]
    pred = (proba >= threshold).astype(int)
    if show:
        print(classification_report(y_, pred, digits=4))
        print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y_, pred))
        print("PR-AUC:", round(average_precision_score(y_, proba), 4), "| ROC-AUC:", round(roc_auc_score(y_, proba), 4))
    return proba

for n, p in pipes.items():
    print("=" * 60, "\n", n, "(validation set)")
    evaluate_model(p, X_val, y_val)

## 5. Comparison table (validation set)
Real measured values only. Select the model here, not on the test set.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for n, p in pipes.items():
    proba = p.predict_proba(X_val)[:, 1]; pred = (proba >= 0.5).astype(int)
    rows.append({"Model": n, "Precision": precision_score(y_val, pred, zero_division=0),
                 "Recall": recall_score(y_val, pred), "F1-score": f1_score(y_val, pred),
                 "PR-AUC": average_precision_score(y_val, proba), "ROC-AUC": roc_auc_score(y_val, proba)})
comparison = pd.DataFrame(rows).set_index("Model").round(4)
comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n, p in pipes.items():
    proba = p.predict_proba(X_val)[:, 1]
    pr, rc, _ = precision_recall_curve(y_val, proba); fpr, tpr, _ = roc_curve(y_val, proba)
    ax[0].plot(rc, pr, label=n); ax[1].plot(fpr, tpr, label=n)
ax[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (validation)")
ax[1].set(xlabel="FPR", ylabel="TPR", title="ROC (validation)"); ax[1].plot([0, 1], [0, 1], "--", c="grey")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. Imbalance experiment: SMOTE on training data only
SMOTE is placed **inside** the imblearn pipeline, so it only ever sees training data. Never apply it before splitting.

In [ ]:
smote_rf = Pipeline([("prep", prep()), ("smote", SMOTE(sampling_strategy=0.1, random_state=42)),
                     ("clf", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=2,
                                                    n_jobs=-1, random_state=42))]).fit(X_train, y_train)
p_cw = pipes["Random Forest"].predict_proba(X_val)[:, 1]; p_sm = smote_rf.predict_proba(X_val)[:, 1]
print("Random Forest PR-AUC  class weights:", round(average_precision_score(y_val, p_cw), 4),
      "| SMOTE:", round(average_precision_score(y_val, p_sm), 4))

## 7. Choose a threshold on validation, then evaluate ONCE on the test set

In [ ]:
best_name = comparison["PR-AUC"].idxmax(); best = pipes[best_name]
print("Selected model (highest validation PR-AUC):", best_name)
pv = best.predict_proba(X_val)[:, 1]
p, r, t = precision_recall_curve(y_val, pv)
f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
thr = float(t[f1.argmax()]); print(f"Threshold maximising validation F1: {thr:.3f}")

# refit on train+validation, then touch the test set once
final = Pipeline([("prep", prep()), ("clf", models[best_name].__class__(**models[best_name].get_params()))]).fit(X_trainval, y_trainval)
print("\nFINAL TEST RESULTS")
_ = evaluate_model(final, X_test, y_test, threshold=thr)

## 8. Save the model for the dashboard
For the complete dashboard artefacts (all three models, figures, JSON), run `python src/train.py`.

In [ ]:
Path("../models").mkdir(exist_ok=True)
joblib.dump({"pipeline": final, "threshold": thr, "features": list(X.columns), "name": best_name,
             "strategy": "class_weight", "synthetic": "demo" in DATA_PATH.name},
            "../models/fraud_model_notebook.joblib", compress=3)
print("saved")

## 9. Conclusions (write in your own words from YOUR numbers)
* Which model had the best PR-AUC and why might that be?
* How many frauds were missed (FN) and how many genuine transactions flagged (FP)?
* Did SMOTE help compared with class weights?
* What are the limitations (few frauds, anonymised features, two days of data)?